In [39]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('TkAgg')
import warnings
warnings.filterwarnings('ignore')
from matplotlib.animation import FuncAnimation, PillowWriter


In [40]:
def rastrigin(x):
    A = 10
    n = len(x)
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))

def rosenbrock_cubic(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def constraint_cubic(x, y):
    return (x - 1)**3 - y + 1 < 0


def mishra_bird(x, y):
    term1 = np.exp((1 - np.cos(x))**2) * np.sin(y)
    term2 = np.exp((1 - np.sin(y))**2) * np.cos(x)
    return term1 + term2 + (x - y)**2

def constraint_mishra(x, y):
    return (x + 5)**2 + (y + 5)**2 < 25


#Штраф
def evaluate(x, y, func, constraint, penalty=1e6):
    if constraint(x, y):
        return func(x, y)
    else:
        return func(x, y) + penalty

In [41]:
# Універсальна оцінка
def evaluate_point(p, func, constraint=None):
    if constraint is None:
        return func(p)
    else:
        return func(p[0], p[1]) if constraint is None else evaluate(p[0], p[1], func, constraint)


def pso(func, bounds, constraint=None,
        particles=30, iterations=100,
        runs=1,
        v_range=(-1,1),
        alpha1=2.0, alpha2=2.0,
        round_index=None):

    dim = len(bounds)
    low = np.array([b[0] for b in bounds])
    high = np.array([b[1] for b in bounds])

    curves = []

    for _ in range(runs):

        pos = np.random.uniform(low, high, (particles, dim))
        vel = np.random.uniform(v_range[0], v_range[1], (particles, dim))

        if round_index is not None:
            pos[:,round_index] = np.round(pos[:,round_index])

        pbest = pos.copy()
        pbest_val = np.array([evaluate_point(p, func, constraint) for p in pos])

        gbest_idx = np.argmin(pbest_val)
        gbest = pbest[gbest_idx].copy()
        gbest_val = pbest_val[gbest_idx]

        history = []

        for _ in range(iterations):

            for i in range(particles):

                r1, r2 = np.random.rand(dim), np.random.rand(dim)

                w = 0.7 
                vel[i] = w * vel[i] + alpha1*r1*(pbest[i]-pos[i]) + alpha2*r2*(gbest-pos[i])
                vel[i] = np.clip(vel[i], v_range[0], v_range[1])

                pos[i] += vel[i]
                pos[i] = np.clip(pos[i], low, high)

                if round_index is not None:
                    pos[i,round_index] = np.round(pos[i,round_index])

                fit = evaluate_point(pos[i], func, constraint)

                if fit < pbest_val[i]:
                    pbest[i] = pos[i].copy()
                    pbest_val[i] = fit

            idx = np.argmin(pbest_val)

            if pbest_val[idx] < gbest_val:
                gbest = pbest[idx].copy()
                gbest_val = pbest_val[idx]

            history.append(gbest_val)

        curves.append(history)

    return np.mean(curves, axis=0)


def bees(func, bounds, constraint=None,
         population=30, iterations=100,
         runs=1,
         elite_sites=3, best_sites=6,
         elite_bees=7, other_bees=3,
         alpha=0.95,
         round_index=None):

    dim = len(bounds)
    low = np.array([b[0] for b in bounds])
    high = np.array([b[1] for b in bounds])

    curves = []

    for _ in range(runs):

        pop = np.random.uniform(low, high, (population, dim))
        neigh_0 = 0.5
        neigh = neigh_0

        if round_index is not None:
            pop[:,round_index] = np.round(pop[:,round_index])

        curve = []

        for iteration in range(iterations):

            fitness = np.array([evaluate_point(p, func, constraint) for p in pop])

            idx = np.argsort(fitness)
            pop = pop[idx]
            fitness = fitness[idx]

            new_pop = []

            for i in range(best_sites):

                site = pop[i].copy()
                recruits = elite_bees if i < elite_sites else other_bees

                best_site = site
                best_fit = fitness[i]

                for _ in range(recruits):

                    candidate = site + neigh*np.random.uniform(-1,1,dim)
                    candidate = np.clip(candidate, low, high)

                    if round_index is not None:
                        candidate[round_index] = np.round(candidate[round_index])

                    fit = evaluate_point(candidate, func, constraint)

                    if fit < best_fit:
                        best_site = candidate
                        best_fit = fit

                new_pop.append(best_site)

            scouts = np.random.uniform(low, high, (population-best_sites, dim))
            pop = np.vstack((new_pop, scouts))

            neigh = neigh_0 * (alpha**iteration)

            curve.append(np.min([evaluate_point(p, func, constraint) for p in pop]))

        curves.append(curve)

    return np.mean(curves, axis=0)


def firefly(func, bounds, constraint=None,
            population=25, iterations=100,
            runs=1,
            beta0=1.0, gamma=0.1, alpha=0.2,
            round_index=None):

    dim = len(bounds)
    low = np.array([b[0] for b in bounds])
    high = np.array([b[1] for b in bounds])

    curves = []

    for _ in range(runs):

        pop = np.random.uniform(low, high, (population, dim))

        if round_index is not None:
            pop[:,round_index] = np.round(pop[:,round_index])

        intensity = np.array([evaluate_point(p, func, constraint) for p in pop])
        curve = []

        for _ in range(iterations):

            for i in range(population):
                for j in range(population):

                    if intensity[j] < intensity[i]:

                        r = np.linalg.norm(pop[i]-pop[j])
                        beta = beta0*np.exp(-gamma*r**2)

                        pop[i] += beta*(pop[j]-pop[i]) + alpha*(np.random.rand(dim)-0.5)
                        pop[i] = np.clip(pop[i], low, high)

                        if round_index is not None:
                            pop[i,round_index] = np.round(pop[i,round_index])

            intensity = np.array([evaluate_point(p, func, constraint) for p in pop])
            curve.append(np.min(intensity))

        curves.append(curve)

    return np.mean(curves, axis=0)

### Завдання 2

In [45]:
func = rastrigin  # використовуємо функцію Растригіна
bounds = [(-5.12, 5.12)] * 15  # межі для максимальної розмірності (15)

dimensions = [2, 5, 10, 15]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, dim in enumerate(dimensions):
    print(f"Розмірність n = {dim}")
    
    # Обмежуємо розмірність
    current_bounds = bounds[:dim]
    
    # Зберігаємо результати
    pso_curve = pso(func, current_bounds, iterations=100, runs=1)
    bees_curve = bees(func, current_bounds, iterations=100, runs=1)
    firefly_curve = firefly(func, current_bounds, iterations=100, runs=1)
    
    axes[idx].plot(pso_curve, label='PSO', linewidth=2)
    axes[idx].plot(bees_curve, label='Bees', linewidth=2)
    axes[idx].plot(firefly_curve, label='Firefly', linewidth=2)
    axes[idx].set_title(f'n = {dim}')
    axes[idx].set_xlabel('Ітерація')
    axes[idx].set_ylabel('Найкраще значення')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Мінімізація функції Растригіна', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Розмірність n = 2
Розмірність n = 5
Розмірність n = 10
Розмірність n = 15


### Завдання 3

In [ ]:
#Штраф
def evaluate(x, y, func, constraint, penalty=1e6):
    if constraint(x, y):
        return func(x, y)
    else:
        return func(x, y) + penalty


def pso(func, constraint, bounds,
        particles=30, iterations=100,
        alpha1=2.0, alpha2=2.0):

    x_min, x_max = bounds[0]
    y_min, y_max = bounds[1]

    v_min, v_max = -0.5, 0.5

    pos = np.random.uniform([x_min, y_min],
                            [x_max, y_max],
                            (particles, 2))
    vel = np.random.uniform(v_min, v_max, (particles, 2))

    pbest = pos.copy()
    pbest_val = np.array([evaluate(p[0], p[1], func, constraint)
                          for p in pos])

    gbest_idx = np.argmin(pbest_val)
    gbest = pbest[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]
    
    history_pos = [pos.copy()]
    history_best = [gbest.copy()]
    history_val = [gbest_val]

    for n in range(iterations):

        for i in range(particles):

            r1 = np.random.rand(2)
            r2 = np.random.rand(2)

            vel[i] = (vel[i]
                      + alpha1 * r1 * (pbest[i] - pos[i])
                      + alpha2 * r2 * (gbest - pos[i]))

            vel[i] = np.clip(vel[i], v_min, v_max)

            pos[i] += vel[i]
            pos[i] = np.clip(pos[i],
                             [x_min, y_min],
                             [x_max, y_max])

            fit = evaluate(pos[i][0], pos[i][1], func, constraint)

            if fit < pbest_val[i]:
                pbest[i] = pos[i].copy()
                pbest_val[i] = fit

        new_idx = np.argmin(pbest_val)
        if pbest_val[new_idx] < gbest_val:
            gbest = pbest[new_idx].copy()
            gbest_val = pbest_val[new_idx]
            
        history_pos.append(pos.copy())
        history_best.append(gbest.copy())
        history_val.append(gbest_val)

    return history_pos, history_best, history_val


def bees(func, constraint, bounds,
         population=30, iterations=100,
         Ls=6, Les=3,
         Ze=7, Zo=3,
         delta=0.1,
         eta_max=1.0,
         alpha=0.95):

    x_min, x_max = bounds[0]
    y_min, y_max = bounds[1]

    pop = np.random.uniform([x_min, y_min],
                            [x_max, y_max],
                            (population, 2))
    
    # Правильна ініціалізація історії
    fitness = np.array([evaluate(p[0], p[1], func, constraint)
                        for p in pop])
    best_idx = np.argmin(fitness)
    
    history_pos = [pop.copy()]
    history_best = [pop[best_idx].copy()]
    history_val = [fitness[best_idx]]

    for n in range(iterations):

        fitness = np.array([evaluate(p[0], p[1], func, constraint)
                            for p in pop])

        idx = np.argsort(fitness)
        pop = pop[idx]
        fitness = fitness[idx]

        eta = eta_max * (alpha ** n)

        for l in range(Ls):

            Z = Ze if l < Les else Zo

            best_site = pop[l].copy()
            best_site_val = fitness[l]

            for _ in range(Z):

                candidate = best_site + \
                            eta * delta * \
                            np.random.uniform(-1, 1, 2)

                candidate = np.clip(candidate,
                                    [x_min, y_min],
                                    [x_max, y_max])

                cand_val = evaluate(candidate[0], candidate[1],
                                    func, constraint)

                if cand_val < best_site_val:
                    best_site = candidate
                    best_site_val = cand_val

            pop[l] = best_site

        pop[Ls:] = np.random.uniform([x_min, y_min],
                                     [x_max, y_max],
                                     (population - Ls, 2))

        fitness = np.array([evaluate(p[0], p[1], func, constraint)
                            for p in pop])
        best_idx = np.argmin(fitness)
        
        history_pos.append(pop.copy())
        history_best.append(pop[best_idx].copy())
        history_val.append(fitness[best_idx])

    return history_pos, history_best, history_val


def firefly(func, constraint, bounds,
            population=40, iterations=100,
            beta0=1.0, gamma=0.3, alpha=0.3):

    x_min, x_max = bounds[0]
    y_min, y_max = bounds[1]

    pop = np.random.uniform([x_min, y_min],
                            [x_max, y_max],
                            (population, 2))
    
    # Правильна ініціалізація історії
    intensity = np.array([evaluate(p[0], p[1], func, constraint)
                          for p in pop])
    best_idx = np.argmin(intensity)
    
    history_pos = [pop.copy()]
    history_best = [pop[best_idx].copy()]
    history_val = [intensity[best_idx]]

    for _ in range(iterations):

        intensity = np.array([evaluate(p[0], p[1], func, constraint)
                              for p in pop])

        for i in range(population):
            for j in range(population):

                if intensity[j] < intensity[i]:

                    r = np.linalg.norm(pop[i] - pop[j])
                    beta = beta0 * np.exp(-gamma * r**2)

                    pop[i] += beta * (pop[j] - pop[i]) \
                              + alpha * (np.random.rand(2) - 0.5)

                    pop[i] = np.clip(pop[i],
                                     [x_min, y_min],
                                     [x_max, y_max])

        intensity = np.array([evaluate(p[0], p[1], func, constraint)
                              for p in pop])
        best_idx = np.argmin(intensity)

        history_pos.append(pop.copy())
        history_best.append(pop[best_idx].copy())
        history_val.append(intensity[best_idx])

    return history_pos, history_best, history_val

#Анімація
def animate_all(func, constraint, bounds, name, filename):

    pso_pos, pso_best, pso_val = pso(func, constraint, bounds)
    bees_pos, bees_best, bees_val = bees(func, constraint, bounds)
    fire_pos, fire_best, fire_val = firefly(func, constraint, bounds)

    x = np.linspace(bounds[0][0], bounds[0][1], 100)
    y = np.linspace(bounds[1][0], bounds[1][1], 100)
    X, Y = np.meshgrid(x, y)

    Z = np.zeros_like(X)
    for i in range(len(y)):
        for j in range(len(x)):
            if constraint(X[i,j], Y[i,j]):
                Z[i,j] = func(X[i,j], Y[i,j])
            else:
                Z[i,j] = np.nan

    fig, axes = plt.subplots(1, 3, figsize=(15,5))
    fig.suptitle(name)

    data = [
        ("PSO", pso_pos, pso_best, pso_val),
        ("Bees", bees_pos, bees_best, bees_val),
        ("Firefly", fire_pos, fire_best, fire_val)
    ]

    points = []
    best_points = []
    info = []

    for ax, (title, pos_hist, best_hist, val_hist) in zip(axes, data):

        ax.set_xlim(bounds[0])
        ax.set_ylim(bounds[1])
        ax.set_title(title)
        ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)

        p = ax.scatter(pos_hist[0][:,0], pos_hist[0][:,1],
                       c='white', edgecolors='black', s=20)

        b, = ax.plot(best_hist[0][0], best_hist[0][1], 'ro', markersize=6)

        t = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        points.append(p)
        best_points.append(b)
        info.append(t)

    frames = len(pso_pos)

    def update(fr):
        for i in range(3):
            points[i].set_offsets(data[i][1][fr])

            best_points[i].set_data(
                [data[i][2][fr][0]],
                [data[i][2][fr][1]]
            )

            info[i].set_text(f'{fr}\n{data[i][3][fr]:.3f}')

        return points + best_points + info


    anim = FuncAnimation(fig, update, frames=frames, interval=100)

    anim.save(filename, writer=PillowWriter(fps=10))

    plt.close(fig)



funcs = [
    ("Функція Розенброка з кубічним обмеженням",
     rosenbrock_cubic, constraint_cubic,
     [(-1.5, 1.5), (-0.5, 2.5)],
     "rosenbrock_cubic.gif"),

    ("Функція Мішри-Берда з круговим обмеженням",
     mishra_bird, constraint_mishra,
     [(-10, 0), (-6.5, 0)],
     "mishra_bird_circle.gif")
]


for name, f, c, b, filename in funcs:
    animate_all(f, c, b, name, filename)

### Завдання 4

In [51]:
# Функція ваги
def reducer_weight(x):
    x1,x2,x3,x4,x5,x6,x7 = x
    return (0.7854*x1*x2**2*(3.3333*x3**2 + 14.9334*x3 - 43.0934)
            - 1.508*x1*(x6**2+x7**2)
            + 7.4777*(x6**3+x7**3)
            + 0.7854*(x4*x6**2 + x5*x7**2))

# Обмеження
def check_constraints(x):
    x1,x2,x3,x4,x5,x6,x7 = x

    g = np.array([
        27/(x1*x2**2*x3)-1,
        397.5/(x1*x2**2*x3**2)-1,
        1.93*x4**3/(x2*x3*x6**4)-1,
        1.93/(x2*x3*x7**4)-1,
        (np.sqrt((745*x4/(x2*x3))**2 + 16.9e6)/(110*x6**3))-1,
        (np.sqrt((745*x5/(x2*x3))**2 + 157.5e6)/(85*x7**3))-1,
        x2*x3/40-1,
        5*x2/x1-1,
        x1/(12*x2)-1,
        (1.5*x6+1.9)/x4-1,
        (1.1*x7+1.9)/x5-1
    ])

    violation = np.sum(np.maximum(0,g))
    ok = np.all(g<=0)

    return ok, violation


def evaluate(x, penalty=1e6):
    ok, viol = check_constraints(x)
    val = reducer_weight(x)

    if ok:
        return val
    else:
        return val + 10000 + viol * 1e5 

# Межі
bounds = [(2.6,3.6),(0.7,0.8),(17,28),
          (7.3,8.3),(7.8,8.3),(2.9,3.9),(5.0,5.5)]

low = np.array([b[0] for b in bounds])
high = np.array([b[1] for b in bounds])
dim = len(bounds)

def pso(func, particles=40, iterations=150):

    pos = np.random.uniform(low, high, (particles, dim))
    pos[:,2] = np.round(pos[:,2])

    vel = np.random.uniform(-0.1,0.1,(particles,dim))

    pbest = pos.copy()
    pbest_val = np.array([func(p) for p in pos])

    gbest_idx = np.argmin(pbest_val)
    gbest = pbest[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]

    history=[gbest_val]

    w,c1,c2 = 0.7,2,2

    w_max, w_min = 0.9, 0.4
    for _ in range(iterations):
        w = w_max - (w_max - w_min) * (i / iterations)

        r1 = np.random.rand(particles,dim)
        r2 = np.random.rand(particles,dim)

        vel = w*vel + c1*r1*(pbest-pos) + c2*r2*(gbest-pos)
        pos += vel

        pos = np.clip(pos,low,high)
        pos[:,2] = np.round(pos[:,2])

        vals = np.array([func(p) for p in pos])

        better = vals < pbest_val
        pbest[better] = pos[better]
        pbest_val[better] = vals[better]

        idx = np.argmin(pbest_val)

        if pbest_val[idx] < gbest_val:
            gbest = pbest[idx].copy()
            gbest_val = pbest_val[idx]

        history.append(gbest_val)

    return gbest, gbest_val, history


def bees(func, population=40, iterations=150):

    pop = np.random.uniform(low,high,(population,dim))
    pop[:,2] = np.round(pop[:,2])

    vals = np.array([func(p) for p in pop])

    best_idx = np.argmin(vals)
    best = pop[best_idx].copy()
    best_val = vals[best_idx]

    history=[best_val]

    for _ in range(iterations):

        for i in range(population):

            candidate = pop[i] + 0.1*(high-low)*(np.random.rand(dim)-0.5)

            candidate = np.clip(candidate,low,high)
            candidate[2] = np.round(candidate[2])

            val = func(candidate)

            if val < vals[i]:
                pop[i] = candidate
                vals[i] = val

        best_idx = np.argmin(vals)

        if vals[best_idx] < best_val:
            best = pop[best_idx].copy()
            best_val = vals[best_idx]

        history.append(best_val)

    return best,best_val,history


def firefly(func, population=35, iterations=150):

    pop = np.random.uniform(low,high,(population,dim))
    pop[:,2] = np.round(pop[:,2])

    vals = np.array([func(p) for p in pop])

    best_idx = np.argmin(vals)
    best = pop[best_idx].copy()
    best_val = vals[best_idx]

    history=[best_val]

    gamma=1
    alpha=0.2

    for _ in range(iterations):

        for i in range(population):
            for j in range(population):

                if vals[j] < vals[i]:

                    r = np.linalg.norm(pop[i]-pop[j])
                    beta = np.exp(-gamma*r*r)

                    pop[i] += beta*(pop[j]-pop[i]) + alpha*(np.random.rand(dim)-0.5)

                    pop[i] = np.clip(pop[i],low,high)
                    pop[i,2] = np.round(pop[i,2])

                    vals[i] = func(pop[i])

        idx=np.argmin(vals)

        if vals[idx] < best_val:
            best=pop[idx].copy()
            best_val=vals[idx]

        history.append(best_val)

    return best,best_val,history



pso_best,pso_val,pso_hist = pso(evaluate)
bees_best,bees_val,bees_hist = bees(evaluate)
fire_best,fire_val,fire_hist = firefly(evaluate)


plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(pso_hist,label="PSO")
plt.plot(bees_hist,label="Bees")
plt.plot(fire_hist,label="Firefly")
plt.yscale("log")
plt.xlabel("Ітерація")
plt.ylabel("Вага")
plt.title("Збіжність алгоритмів")
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)

algos=["PSO","Bees","Firefly"]
vals=[pso_val,bees_val,fire_val]

colors=["tab:blue","tab:orange","tab:green"]

plt.bar(algos,vals,color=colors)

for i,v in enumerate(vals):
    plt.text(i,v+1,f"{v:.2f}",ha="center")

plt.title("Фінальні значення")
plt.ylabel("Вага")

plt.tight_layout()
plt.savefig("Task4.png")

print("\nРезультати оптимізації")

print("\nPSO:")
print(f"x1={pso_best[0]:.4f}, x2={pso_best[1]:.4f}, x3={pso_best[2]:.4f}, "
      f"x4={pso_best[3]:.4f}, x5={pso_best[4]:.4f}, x6={pso_best[5]:.4f}, x7={pso_best[6]:.4f}")
print(f"f(x) = {pso_val:.4f}")

print("\nBees:")
print(f"x1={bees_best[0]:.4f}, x2={bees_best[1]:.4f}, x3={bees_best[2]:.4f}, "
      f"x4={bees_best[3]:.4f}, x5={bees_best[4]:.4f}, x6={bees_best[5]:.4f}, x7={bees_best[6]:.4f}")
print(f"f(x) = {bees_val:.4f}")

print("\nFirefly:")
print(f"x1={fire_best[0]:.4f}, x2={fire_best[1]:.4f}, x3={fire_best[2]:.4f}, "
      f"x4={fire_best[3]:.4f}, x5={fire_best[4]:.4f}, x6={fire_best[5]:.4f}, x7={fire_best[6]:.4f}")
print(f"f(x) = {fire_val:.4f}")


Результати оптимізації

PSO:
x1=3.6000, x2=0.7000, x3=17.0000, x4=7.3000, x5=8.3000, x6=3.3502, x7=5.2880
f(x) = 3047.4220

Bees:
x1=3.5039, x2=0.7000, x3=17.0000, x4=7.7630, x5=7.9769, x6=3.3768, x7=5.2959
f(x) = 3018.6640

Firefly:
x1=3.5289, x2=0.7000, x3=17.0000, x4=7.9846, x5=7.9992, x6=3.7139, x7=5.4267
f(x) = 3213.9848


### Завдання 5

In [52]:
# Цільова функція
def spring_weight(x):
    x1, x2, x3 = x
    return (x3 + 2) * x2 * x1**2

#Обмеження
def check_constraints(x):
    x1, x2, x3 = x
    g = np.array([
        1 - (x2**3 * x3) / (7.178 * x1**4),
        ((4*x2**2 - x1*x2) /
        (12.566*(x2*x1**3) - x1**4)) +
        1/(5.108*x1**2) - 1,
        1 - (140.45 * x1) / (x2**2 * x3),
        (x2 + x1)/1.5 - 1
    ])

    violation = np.sum(np.maximum(0, g))
    ok = np.all(g <= 0)
    return ok, violation

# Оцінка/Штраф
def evaluate(x, penalty=1e5):

    ok, viol = check_constraints(x)
    val = spring_weight(x)

    if ok:
        return val
    else:
        return val + penalty*(1 + viol)


# Межі змінних
bounds = [
    (0.005, 2.0),
    (0.25, 1.3),
    (2.0, 15.0)
]

low = np.array([b[0] for b in bounds])
high = np.array([b[1] for b in bounds])
dim = len(bounds)


def pso(func, particles=40, iterations=150):

    pos = np.random.uniform(low, high, (particles, dim))
    vel = np.random.uniform(-0.1, 0.1, (particles, dim))

    pbest = pos.copy()
    pbest_val = np.array([func(p) for p in pos])

    gbest_idx = np.argmin(pbest_val)
    gbest = pbest[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]

    history = [gbest_val]

    w, c1, c2 = 0.7, 2, 2

    for _ in range(iterations):

        r1 = np.random.rand(particles, dim)
        r2 = np.random.rand(particles, dim)

        vel = w*vel + c1*r1*(pbest-pos) + c2*r2*(gbest-pos)
        pos += vel

        pos = np.clip(pos, low, high)

        vals = np.array([func(p) for p in pos])

        better = vals < pbest_val
        pbest[better] = pos[better]
        pbest_val[better] = vals[better]

        idx = np.argmin(pbest_val)

        if pbest_val[idx] < gbest_val:
            gbest = pbest[idx].copy()
            gbest_val = pbest_val[idx]

        history.append(gbest_val)

    return gbest, gbest_val, history


def bees(func, population=40, iterations=150):

    pop = np.random.uniform(low, high, (population, dim))
    vals = np.array([func(p) for p in pop])

    best_idx = np.argmin(vals)
    best = pop[best_idx].copy()
    best_val = vals[best_idx]

    history = [best_val]

    for _ in range(iterations):

        for i in range(population):

            candidate = pop[i] + 0.1*(high-low)*(np.random.rand(dim)-0.5)

            candidate = np.clip(candidate, low, high)

            val = func(candidate)

            if val < vals[i]:
                pop[i] = candidate
                vals[i] = val

        best_idx = np.argmin(vals)

        if vals[best_idx] < best_val:
            best = pop[best_idx].copy()
            best_val = vals[best_idx]

        history.append(best_val)

    return best, best_val, history


def firefly(func, population=35, iterations=150):

    pop = np.random.uniform(low, high, (population, dim))
    vals = np.array([func(p) for p in pop])

    best_idx = np.argmin(vals)
    best = pop[best_idx].copy()
    best_val = vals[best_idx]

    history = [best_val]

    gamma = 1
    alpha = 0.2

    for _ in range(iterations):

        for i in range(population):
            for j in range(population):

                if vals[j] < vals[i]:

                    r = np.linalg.norm(pop[i]-pop[j])
                    beta = np.exp(-gamma*r*r)

                    pop[i] += beta*(pop[j]-pop[i]) + alpha*(np.random.rand(dim)-0.5)

                    pop[i] = np.clip(pop[i], low, high)

                    vals[i] = func(pop[i])

        idx = np.argmin(vals)

        if vals[idx] < best_val:
            best = pop[idx].copy()
            best_val = vals[idx]

        history.append(best_val)

    return best, best_val, history


pso_best, pso_val, pso_hist = pso(evaluate)
bees_best, bees_val, bees_hist = bees(evaluate)
fire_best, fire_val, fire_hist = firefly(evaluate)

print("\nРезультати оптимізації пружини")

print("\nPSO:")
print(f"x1 (діаметр дроту) = {pso_best[0]:.6f}")
print(f"x2 (діаметр котушки) = {pso_best[1]:.6f}")
print(f"x3 (кількість витків) = {pso_best[2]:.6f}")
print(f"f(x) = {pso_val:.6f}")

print("\nBees:")
print(f"x1 (діаметр дроту) = {bees_best[0]:.6f}")
print(f"x2 (діаметр котушки) = {bees_best[1]:.6f}")
print(f"x3 (кількість витків) = {bees_best[2]:.6f}")
print(f"f(x) = {bees_val:.6f}")

print("\nFirefly:")
print(f"x1 (діаметр дроту) = {fire_best[0]:.6f}")
print(f"x2 (діаметр котушки) = {fire_best[1]:.6f}")
print(f"x3 (кількість витків) = {fire_best[2]:.6f}")
print(f"f(x) = {fire_val:.6f}")


plt.figure(figsize=(12,5))
plt.subplot(1,2,1)

plt.plot(pso_hist,label="PSO")
plt.plot(bees_hist,label="Bees")
plt.plot(fire_hist,label="Firefly")

plt.yscale("log")

plt.xlabel("Ітерація")
plt.ylabel("Значення функції")
plt.title("Збіжність алгоритмів")
plt.legend()
plt.grid(True)


# Фінальні значення
plt.subplot(1,2,2)

algos=["PSO","Bees","Firefly"]
vals=[pso_val,bees_val,fire_val]

colors=["tab:blue","tab:orange","tab:green"]

plt.bar(algos,vals,color=colors,edgecolor="black")

for i,v in enumerate(vals):
    plt.text(i,v+0.0005,f"{v:.5f}",ha="center")

plt.title("Фінальні значення")
plt.ylabel("Значення функції")

plt.tight_layout()
plt.savefig("Task5.png")



Результати оптимізації пружини

PSO:
x1 (діаметр дроту) = 0.698566
x2 (діаметр котушки) = 0.769576
x3 (кількість витків) = 3.752964
f(x) = 2.160517

Bees:
x1 (діаметр дроту) = 0.662320
x2 (діаметр котушки) = 0.620933
x3 (кількість витків) = 5.772264
f(x) = 2.117036

Firefly:
x1 (діаметр дроту) = 0.651874
x2 (діаметр котушки) = 0.589667
x3 (кількість витків) = 6.352352
f(x) = 2.092873
